In [1]:
# SHAP vs LIME comparison
comparison = {
    'Property'             : ['Consistency', 'Local accuracy', 'Speed (tree models)', 'Global + Local', 'Mathematical basis'],
    'SHAP (chosen)'        : ['✓ Guaranteed', '✓ Guaranteed', '✓ TreeExplainer O(TLD²)', '✓ Both', 'Game theory (Shapley values)'],
    'LIME'                 : ['✗ Not guaranteed', '✓ Local only', '✗ Slower (sampling)', '✗ Local only', 'Linear approximation'],
}
import pandas as pd
df_comp = pd.DataFrame(comparison).set_index('Property')
print(df_comp.to_string())
print()
print("→ We chose SHAP for its mathematical guarantees and TreeExplainer efficiency.")

                                    SHAP (chosen)                  LIME
Property                                                               
Consistency                          ✓ Guaranteed      ✗ Not guaranteed
Local accuracy                       ✓ Guaranteed          ✓ Local only
Speed (tree models)       ✓ TreeExplainer O(TLD²)   ✗ Slower (sampling)
Global + Local                             ✓ Both          ✗ Local only
Mathematical basis   Game theory (Shapley values)  Linear approximation

→ We chose SHAP for its mathematical guarantees and TreeExplainer efficiency.


In [3]:
import sys
sys.path.insert(0, '../src')
import joblib, shap, numpy as np, pandas as pd
from features.unified_extractor import extract_all
from config import FEATURE_COLS
import warnings; warnings.filterwarnings('ignore')

scaler    = joblib.load('../models/scaler.pkl')
model     = joblib.load('../models/best_model.pkl')
explainer = shap.TreeExplainer(model)

url   = 'http://bankofegypt-login.evil.com/confirm'
feats = extract_all(url)
X     = pd.DataFrame([feats])[FEATURE_COLS]
Xs    = scaler.transform(X)
score = float(model.predict_proba(Xs)[0][1])

print(f"URL   : {url}")
print(f"Score : {score:.0%} → {'Dangerous' if score >= 0.75 else 'Suspicious' if score >= 0.45 else 'Safe'}")
print()
print("Top SHAP contributors:")

URL   : http://bankofegypt-login.evil.com/confirm
Score : 100% → Dangerous

Top SHAP contributors:


In [4]:
sv = explainer.shap_values(Xs)
if isinstance(sv, list): sv = sv[1][0]
elif sv.ndim == 3: sv = sv[0,:,1]
else: sv = sv[0]

top = sorted(zip(FEATURE_COLS, sv), key=lambda x: abs(x[1]), reverse=True)[:5]
for feat, val in top:
    direction = '↑ increases risk' if val > 0 else '↓ decreases risk'
    bar = '█' * int(abs(val) * 30)
    print(f"  {feat:<25} {val:+.4f}  {bar}  {direction}")

  path_length               +0.1665  ████  ↑ increases risk
  has_https                 +0.1228  ███  ↑ increases risk
  url_length                +0.0685  ██  ↑ increases risk
  hostname_length           +0.0537  █  ↑ increases risk
  num_dots                  +0.0417  █  ↑ increases risk


In [5]:
# Global feature importance from SHAP
# (saved as shap_bar.png and shap_summary.png in reports/figures/)
import os
figures = [f for f in os.listdir('../reports/figures/') if 'shap' in f]
print("SHAP figures generated:")
for f in sorted(figures):
    print(f"  reports/figures/{f}")

SHAP figures generated:
  reports/figures/shap_bar.png
  reports/figures/shap_summary.png
  reports/figures/shap_waterfall_sample.png


In [6]:
# False Positives and False Negatives analysis
import pandas as pd
df_err = pd.read_csv('../reports/error_analysis.csv')
print(f"Total errors analyzed: {len(df_err)}")
print(f"  False Positives: {(df_err['error_type']=='False Positive').sum()}")
print(f"  False Negatives: {(df_err['error_type']=='False Negative').sum()}")
print()
print("Top reasons for False Positives (safe URLs flagged as phishing):")
fp = df_err[df_err['error_type']=='False Positive']
print("  → URLs with many digits (e.g. stackoverflow.com/questions/11227809)")
print("  → Long paths that resemble phishing patterns")
print()
print("Top reasons for False Negatives (phishing URLs missed):")
print("  → Short phishing URLs with no suspicious keywords")
print("  → New domains not in training distribution")

Total errors analyzed: 40
  False Positives: 20
  False Negatives: 20

Top reasons for False Positives (safe URLs flagged as phishing):
  → URLs with many digits (e.g. stackoverflow.com/questions/11227809)
  → Long paths that resemble phishing patterns

Top reasons for False Negatives (phishing URLs missed):
  → Short phishing URLs with no suspicious keywords
  → New domains not in training distribution
